# 01 · City profile and market data (real, official statistics)

**Goal:** one clean row per city with population, income, pay and retail output, plus one tidy table of online shopping share for the UK, US and Taiwan.
These real numbers are used in notebook 02 to calibrate the simulated store data, and in the Power BI market page.

| City set | Source |
|---|---|
| 10 largest UK retail cities (GlobalData definition) + Belfast, each measured as its **city-region** | ONS, NRS, NISRA via ONS tables |
| Dublin (Dublin region, 4 councils) | Central Statistics Office (CSO), Ireland |

In [1]:
import os, glob, zipfile
import pandas as pd
import numpy as np

RAW, CLEAN = "data/raw", "data/clean"
os.makedirs(CLEAN, exist_ok=True)

# Each city is measured as its CITY-REGION (the core city plus the councils around it that form one
# economic area), so that every city is compared on the same kind of geography.
# London = Greater London (33 boroughs); Manchester = the 10 Greater Manchester boroughs; and so on.
# Membership follows the official combined authorities (England) and city-region deals (Scotland, Wales, NI).
CITY_REGIONS = {
    # city: (nation, area name, core council for median pay (ASHE code), member councils as named by ONS)
    "London":     ("England", "Greater London (33 boroughs)", "E12000007", "LONDON"),
    "Birmingham": ("England", "West Midlands Combined Authority", "E08000025",
                   ["Birmingham", "Coventry", "Dudley", "Sandwell", "Solihull", "Walsall", "Wolverhampton"]),
    "Manchester": ("England", "Greater Manchester Combined Authority", "E08000003",
                   ["Bolton", "Bury", "Manchester", "Oldham", "Rochdale", "Salford", "Stockport", "Tameside", "Trafford", "Wigan"]),
    "Leeds":      ("England", "West Yorkshire Combined Authority", "E08000035",
                   ["Leeds", "Bradford", "Calderdale", "Kirklees", "Wakefield"]),
    "Liverpool":  ("England", "Liverpool City Region Combined Authority", "E08000012",
                   ["Liverpool", "Knowsley", "Sefton", "St. Helens", "Wirral", "Halton"]),
    "Sheffield":  ("England", "South Yorkshire Mayoral Combined Authority", "E08000039",
                   ["Sheffield", "Barnsley", "Doncaster", "Rotherham"]),
    "Bristol":    ("England", "West of England Combined Authority", "E06000023",
                   ["Bristol, City of", "Bath and North East Somerset", "South Gloucestershire"]),
    "Glasgow":    ("Scotland", "Glasgow City Region", "S12000049",
                   ["Glasgow City", "East Dunbartonshire", "East Renfrewshire", "Inverclyde", "North Lanarkshire",
                    "Renfrewshire", "South Lanarkshire", "West Dunbartonshire"]),
    "Edinburgh":  ("Scotland", "Edinburgh and South East Scotland City Region", "S12000036",
                   ["City of Edinburgh", "East Lothian", "Midlothian", "West Lothian", "Fife", "Scottish Borders"]),
    "Cardiff":    ("Wales", "Cardiff Capital Region", "W06000015",
                   ["Cardiff", "Blaenau Gwent", "Bridgend", "Caerphilly", "Merthyr Tydfil", "Monmouthshire", "Newport",
                    "Rhondda Cynon Taf", "Torfaen", "Vale of Glamorgan"]),
    "Belfast":    ("Northern Ireland", "Belfast Region City Deal", "N92000002",   # ASHE has no Belfast figure -> NI average
                   ["Belfast", "Antrim and Newtownabbey", "Ards and North Down", "Lisburn and Castlereagh",
                    "Mid and East Antrim", "Newry, Mourne and Down"]),
}

# Turn council names into ONS codes using the population table (one row per council)
pop_all = pd.read_excel(f"{RAW}/uk_population_mid2025.xlsx", sheet_name="MYE2 - Persons", header=7)
councils = pop_all[pop_all["Geography"].isin(["Unitary Authority", "Metropolitan District", "Non-metropolitan District",
                                              "London Borough", "Council Area", "Local Government District"])]
rows = []
for city, (nation, area, ashe_code, members) in CITY_REGIONS.items():
    if members == "LONDON":
        m = councils[councils["Geography"] == "London Borough"]
    else:
        m = councils[councils["Name"].isin(members)]
        assert len(m) == len(members), f"{city}: found {len(m)} of {len(members)} councils"
    for _, r in m.iterrows():
        rows.append({"city": city, "nation": nation, "area": area, "ashe_code": ashe_code,
                     "la_code": r["Code"], "la_name": r["Name"], "population": r["All ages"]})
members = pd.DataFrame(rows)
members.groupby(["city", "area"]).agg(councils=("la_code", "size"), population=("population", "sum"))

,,councils,population
city,area,,
Belfast,Belfast Region City Deal,6,1141036
Birmingham,West Midlands Combined Authority,7,3027190
Bristol,West of England Combined Authority,3,1004945
Cardiff,Cardiff Capital Region,10,1564798
Edinburgh,Edinburgh and South East Scotland City Region,6,1426250
Glasgow,Glasgow City Region,8,1898140
Leeds,West Yorkshire Combined Authority,5,2424324
Liverpool,Liverpool City Region Combined Authority,6,1610919
London,Greater London (33 boroughs),33,9122909


## 1. Population, mid-2025 (ONS)

The population of each city-region is the sum of its councils (already in the table above).

In [2]:
cities = members.groupby(["city", "nation", "area", "ashe_code"], as_index=False, sort=False)["population"].sum()
cities[["city", "area", "population"]]

,city,area,population
0,London,Greater London (33 boroughs),9122909
1,Birmingham,West Midlands Combined Authority,3027190
2,Manchester,Greater Manchester Combined Authority,3017590
3,Leeds,West Yorkshire Combined Authority,2424324
4,Liverpool,Liverpool City Region Combined Authority,1610919
5,Sheffield,South Yorkshire Mayoral Combined Authority,1434478
6,Bristol,West of England Combined Authority,1004945
7,Glasgow,Glasgow City Region,1898140
8,Edinburgh,Edinburgh and South East Scotland City Region,1426250
9,Cardiff,Cardiff Capital Region,1564798


## 2. Income: gross disposable household income (GDHI) per head, 2024 (ONS)

For each city-region I add up total GDHI and population across its councils, then divide.

In [3]:
def read_gdhi(table):
    t = pd.read_excel(f"{RAW}/uk_gdhi_local_authorities_1997_2024.xlsx", sheet_name=table, header=3)
    t.columns = [str(c).replace(".0", "") for c in t.columns]
    return t.rename(columns={"LAD code": "la_code"})[["la_code", "2024"]]

gdhi_la = read_gdhi("Table 1").rename(columns={"2024": "gdhi_gbp_m"}).merge(
          read_gdhi("Table 2").rename(columns={"2024": "gdhi_pop"}), on="la_code")
members = members.merge(gdhi_la, on="la_code", how="left")
assert members["gdhi_gbp_m"].notna().all(), members[members["gdhi_gbp_m"].isna()]

g = members.groupby("city").agg(gdhi=("gdhi_gbp_m", "sum"), gpop=("gdhi_pop", "sum"))
cities["gdhi_per_head"] = cities["city"].map(g["gdhi"] * 1e6 / g["gpop"])
cities[["city", "gdhi_per_head"]].round(0)

,city,gdhi_per_head
0,London,36487.0
1,Birmingham,19687.0
2,Manchester,21730.0
3,Leeds,21158.0
4,Liverpool,21535.0
5,Sheffield,20556.0
6,Bristol,26417.0
7,Glasgow,22634.0
8,Edinburgh,26793.0
9,Cardiff,21525.0


## 3. Pay: median gross annual pay, 2025 provisional (ONS ASHE Table 8, by home location)

A median cannot be added up across councils, so this column shows the **core city's** median pay (Northern Ireland average for Belfast). It is used only as a profile figure.

In [4]:
with zipfile.ZipFile(f"{RAW}/uk_ashe_table8_2025_provisional.zip") as z:
    name = [n for n in z.namelist() if "8.7a" in n][0]          # Table 8.7a = annual pay, gross
    with z.open(name) as f:
        ashe = pd.read_excel(f, sheet_name="All", header=4)

ashe = ashe[["Code", "Median"]].rename(columns={"Code": "ashe_code", "Median": "median_pay"})
ashe["median_pay"] = pd.to_numeric(ashe["median_pay"], errors="coerce")
cities = cities.merge(ashe, on="ashe_code", how="left")
cities[["city", "median_pay"]]

,city,median_pay
0,London,39778.0
1,Birmingham,30180.0
2,Manchester,30067.0
3,Leeds,31679.0
4,Liverpool,30947.0
5,Sheffield,31816.0
6,Bristol,34051.0
7,Glasgow,32121.0
8,Edinburgh,37380.0
9,Cardiff,32756.0


## 4. Retail output: GVA of retail trade (SIC 47), 2024, current prices (ONS)

ONS **suppresses** some council figures (shown as `[u]`), for example most of the Glasgow city-region.
I keep them as missing here and estimate them council by council in step 6, recording how much of each city-region's figure is estimated.

In [5]:
gva_rows = []
for f in sorted(glob.glob(f"{RAW}/uk_gva_*.xlsx")):
    t = pd.read_excel(f, sheet_name="Table 3", header=1)
    t.columns = [str(c).replace(".0", "") for c in t.columns]
    t = t[t["SIC07"].astype(str).str.strip() == "47"]               # division 47 = retail trade
    gva_rows.append(t[["LA code", "2024"]])
gva = pd.concat(gva_rows).rename(columns={"LA code": "la_code", "2024": "retail_gva_gbp_m"})
gva["retail_gva_gbp_m"] = pd.to_numeric(gva["retail_gva_gbp_m"], errors="coerce")   # "[u]" -> NaN
members = members.merge(gva, on="la_code", how="left")
members.groupby("city")["retail_gva_gbp_m"].agg(published="count", suppressed=lambda s: s.isna().sum())

,published,suppressed
city,,
Belfast,6,0
Birmingham,7,0
Bristol,0,3
Cardiff,0,10
Edinburgh,6,0
Glasgow,1,7
Leeds,5,0
Liverpool,6,0
London,33,0


## 5. Dublin (CSO)

Dublin figures come from Ireland's CSO and are not identical in definition or year:
population is the Dublin region in April 2026; disposable income per person is for 2023 in euro.
I convert euro to sterling at the 2023 average rate (about £0.87 per €1) so the cities can be compared.

In [6]:
EUR_TO_GBP_2023 = 0.87
ie = pd.read_csv(f"{RAW}/ireland_manual_inputs.csv").set_index("measure")["value"]
dublin = {"city": "Dublin", "nation": "Ireland", "area": "Dublin region (4 councils)", "ashe_code": None,
          "population": ie["population"],
          "gdhi_per_head": ie["disposable_income_per_person"] * EUR_TO_GBP_2023,
          "median_pay": np.nan}
cities = pd.concat([cities, pd.DataFrame([dublin])], ignore_index=True)

## 6. Derived measures and estimates for missing retail output

- **Spending power** = population × disposable income per head. It shows how much money residents have.
- **Retail GVA per head** shows how much retail activity a city-region actually hosts.
- **Retail pull index** = a city-region's share of retail GVA ÷ its share of spending power. Above 1 means it attracts shoppers from outside (a regional shopping hub).

Where ONS suppressed a council's retail GVA, I estimate it as that council's spending power × the median retail-GVA-to-spending-power ratio of all published councils in this study, and record the estimated share.
Dublin has no comparable retail GVA, so its whole figure is estimated the same way.

In [7]:
members["spending_power_gbp_m"] = members["gdhi_gbp_m"] * members["population"] / members["gdhi_pop"]
ratio = (members["retail_gva_gbp_m"] / members["spending_power_gbp_m"]).median()
members["gva_estimated"] = members["retail_gva_gbp_m"].isna()
members["retail_gva_gbp_m"] = members["retail_gva_gbp_m"].fillna(members["spending_power_gbp_m"] * ratio)
print(f"Median retail GVA / spending power (used for estimates): {ratio:.3f}")

r = members.groupby("city", sort=False).agg(retail_gva_gbp_m=("retail_gva_gbp_m", "sum"),
        est_gva=("retail_gva_gbp_m", lambda s: s[members.loc[s.index, "gva_estimated"]].sum()),
        councils=("la_code", "size"), councils_estimated=("gva_estimated", "sum"))
cities = cities.merge(r, on="city", how="left")

# Dublin: all retail GVA estimated
d = cities["city"] == "Dublin"
cities["spending_power_gbp_m"] = cities["population"] * cities["gdhi_per_head"] / 1e6
cities.loc[d, "retail_gva_gbp_m"] = cities.loc[d, "spending_power_gbp_m"] * ratio
cities.loc[d, "est_gva"] = cities.loc[d, "retail_gva_gbp_m"]

cities["retail_gva_estimated_pct"] = (cities["est_gva"] / cities["retail_gva_gbp_m"] * 100).round(1)
cities["retail_gva_is_estimate"] = cities["retail_gva_estimated_pct"] > 0
cities["retail_gva_per_head"] = cities["retail_gva_gbp_m"] * 1e6 / cities["population"]
cities["retail_pull_index"] = (cities["retail_gva_gbp_m"] / cities["retail_gva_gbp_m"].sum()) / \
                              (cities["spending_power_gbp_m"] / cities["spending_power_gbp_m"].sum())

# Keep the same columns as before so the SQL scripts load it unchanged; the area definitions go in their own file.
cities["code"] = cities["ashe_code"].where(cities["city"] != "Belfast", "N09000003")
cities.loc[d, "code"] = "IE-DUBLIN"
city_profile = cities[["city", "nation", "code", "population", "gdhi_per_head", "median_pay", "retail_gva_gbp_m",
                       "spending_power_gbp_m", "retail_gva_is_estimate", "retail_gva_per_head", "retail_pull_index"]]
city_profile.round(4).to_csv(f"{CLEAN}/city_profile.csv", index=False, lineterminator="\n")

areas = cities[["city", "area", "population", "councils", "councils_estimated", "retail_gva_estimated_pct"]]
areas = areas.astype({"councils": "Int64", "councils_estimated": "Int64"})
areas.to_csv(f"{CLEAN}/city_region_definitions.csv", index=False, lineterminator="\n")
members[["city", "la_code", "la_name", "population", "gva_estimated"]].to_csv(
    f"{CLEAN}/city_region_councils.csv", index=False, lineterminator="\n")
cities.sort_values("spending_power_gbp_m", ascending=False)[
    ["city", "area", "population", "spending_power_gbp_m", "retail_gva_gbp_m", "retail_gva_estimated_pct", "retail_pull_index"]].round(2)

Median retail GVA / spending power (used for estimates): 0.064


,city,area,population,spending_power_gbp_m,retail_gva_gbp_m,retail_gva_estimated_pct,retail_pull_index
0,London,Greater London (33 boroughs),9122909,332868.39,21938.00,0.0,0.94
2,Manchester,Greater Manchester Combined Authority,3017590,65572.23,5819.00,0.0,1.26
1,Birmingham,West Midlands Combined Authority,3027190,59597.64,4673.00,0.0,1.11
3,Leeds,West Yorkshire Combined Authority,2424324,51293.13,3993.00,0.0,1.11
11,Dublin,Dublin region (4 councils),1597700,45026.24,2884.50,100.0,0.91
7,Glasgow,Glasgow City Region,1898140,42962.06,2675.58,83.9,0.88
8,Edinburgh,Edinburgh and South East Scotland City Region,1426250,38213.43,1936.00,0.0,0.72
4,Liverpool,Liverpool City Region Combined Authority,1610919,34690.59,2393.00,0.0,0.98
9,Cardiff,Cardiff Capital Region,1564798,33682.09,2158.17,100.0,0.91
5,Sheffield,South Yorkshire Mayoral Combined Authority,1434478,29487.24,2033.00,0.0,0.98


## 7. Online share of retail: UK, US and Taiwan

In [8]:
# UK: ONS series J4MC, monthly, % of retail sales made online (Great Britain)
uk = pd.read_csv(f"{RAW}/uk_internet_share_of_retail_J4MC.csv", header=None, names=["period", "value"])
uk = uk[uk["period"].str.match(r"^\d{4} [A-Z]{3}$", na=False)].copy()
uk["date"] = pd.to_datetime(uk["period"], format="%Y %b")
uk = uk.assign(market="UK", online_share_pct=pd.to_numeric(uk["value"]))[["market", "date", "online_share_pct"]]
uk_q = uk.set_index("date")["online_share_pct"].resample("QS").agg(["mean", "count"])
uk_q = uk_q[uk_q["count"] == 3]["mean"].rename("online_share_pct").reset_index().assign(market="UK")   # complete quarters only

# US: Census e-commerce share of retail, quarterly, seasonally adjusted (FRED ECOMPCTSA)
us = pd.read_csv(f"{RAW}/us_ecommerce_share_of_retail_ECOMPCTSA.csv")
us.columns = ["date", "online_share_pct"]
us = us.assign(market="US", date=pd.to_datetime(us["date"]))

# Taiwan: MOEA online retail sales ÷ total retail sales, quarterly (NT$100 million)
def read_moea(path):
    t = pd.read_excel(path, header=None)
    t = t[t[2].apply(lambda v: isinstance(v, (int, float)) and not pd.isna(v))].copy()
    t["year"] = t[0].ffill().astype(str).str.extract(r"(\d+)")[0].astype(int) + 1911   # ROC year -> AD
    return t

tw_on = read_moea(f"{RAW}/tw_online_retail_sales.xls")
tw_on = tw_on[tw_on[1].astype(str).str.contains("季")]
tw_on["date"] = pd.PeriodIndex.from_fields(year=tw_on["year"], quarter=tw_on[1].str.extract(r"(\d)")[0].astype(int), freq="Q").to_timestamp()
tw_on = tw_on[["date", 2]].rename(columns={2: "online_sales"})

tw_all = read_moea(f"{RAW}/tw_retail_sales.xls")
tw_all = tw_all[tw_all[1].astype(str).str.contains("月")]
tw_all["month"] = tw_all[1].str.extract(r"(\d+)")[0].astype(int)
tw_all["date"] = pd.to_datetime(dict(year=tw_all["year"], month=tw_all["month"], day=1))
tw_q = tw_all.set_index("date")[2].astype(float).resample("QS").agg(["sum", "count"])
tw_q = tw_q[tw_q["count"] == 3]["sum"].rename("retail_sales").reset_index()   # only complete quarters

tw = tw_on.merge(tw_q, on="date")
tw = tw.assign(market="Taiwan", online_share_pct=tw["online_sales"] / tw["retail_sales"] * 100)

online = pd.concat([uk_q, us, tw[["market", "date", "online_share_pct"]]], ignore_index=True)
online["quarter"] = online["date"].dt.to_period("Q").astype(str)
online = online[["market", "quarter", "date", "online_share_pct"]].round(2)
online.to_csv(f"{CLEAN}/online_share_by_market.csv", index=False, lineterminator="\n")
online.pivot(index="quarter", columns="market", values="online_share_pct").tail(8)

/tmp/ipykernel_190/1257439725.py:38: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  online = online[["market", "quarter", "date", "online_share_pct"]].round(2)


market,Taiwan,UK,US
quarter,,,
2024Q3,12.886788,26.466667,16.2
2024Q4,14.428128,29.0,16.2
2025Q1,13.070225,26.433333,16.0
2025Q2,13.460893,26.266667,16.3
2025Q3,14.041424,26.7,16.4
2025Q4,14.786179,29.966667,16.7
2026Q1,14.154357,28.166667,17.0
2026Q2,14.214277,27.9,17.1


## 8. Taiwan: which product categories sell most online?

Both MOEA tables split sales by the same retail categories, so online sales ÷ total sales gives the **online penetration** of each category.
I use the latest four complete quarters that appear in both tables.

In [9]:
CATEGORIES = {   # column position in the MOEA tables -> English name (same position in both files)
    3: "General merchandise", 4: "Food, drink & tobacco", 5: "Clothing & textiles",
    6: "Household goods", 7: "Pharmacy & cosmetics", 8: "Books, toys & leisure",
}

def by_category(path, freq_word):
    t = read_moea(path)
    t = t[t[1].astype(str).str.contains(freq_word)].copy()
    n = t[1].str.extract(r"(\d+)")[0].astype(int)
    if freq_word == "季":
        t["date"] = pd.PeriodIndex.from_fields(year=t["year"], quarter=n, freq="Q").to_timestamp()
    else:
        t["date"] = pd.to_datetime(dict(year=t["year"], month=n, day=1))
    return t.set_index("date")[list(CATEGORIES)].astype(float).rename(columns=CATEGORIES)

online_cat = by_category(f"{RAW}/tw_online_retail_sales.xls", "季")
retail_cat = by_category(f"{RAW}/tw_retail_sales.xls", "月").resample("QS").sum(min_count=3).dropna()

common = online_cat.index.intersection(retail_cat.index)[-4:]
tw_cat = pd.DataFrame({
    "online_sales_nt100m": online_cat.loc[common].sum(),
    "total_sales_nt100m": retail_cat.loc[common].sum(),
})
tw_cat["online_penetration_pct"] = tw_cat["online_sales_nt100m"] / tw_cat["total_sales_nt100m"] * 100
tw_cat["period"] = f"{common.min().to_period('Q')} to {common.max().to_period('Q')}" if len(common) else "n/a"
tw_cat.index.name = "category"
tw_cat.round(1).to_csv(f"{CLEAN}/taiwan_online_penetration_by_category.csv", lineterminator="\n")
print(f"Quarters used: {len(common)}")
tw_cat.sort_values("online_penetration_pct", ascending=False).round(1)

Quarters used: 4


,online_sales_nt100m,total_sales_nt100m,online_penetration_pct,period
category,,,,
"Books, toys & leisure",102.9,928.6,11.1,2025Q3 to 2026Q2
Clothing & textiles,357.6,3549.2,10.1,2025Q3 to 2026Q2
Pharmacy & cosmetics,163.4,2212.6,7.4,2025Q3 to 2026Q2
Household goods,145.7,2348.4,6.2,2025Q3 to 2026Q2
"Food, drink & tobacco",162.1,3161.7,5.1,2025Q3 to 2026Q2
General merchandise,474.1,16337.4,2.9,2025Q3 to 2026Q2


## 9. Market size and growth: UK, US and Taiwan

Market potential is usually judged on four things: **size**, **growth**, **online share** and **distance**. Sections 7 and 8 cover online share; this section adds size and growth.

| Market | Series | Publisher |
|---|---|---|
| UK | Average weekly value for all retailing, £m (JE4W), monthly; × weeks in each month, summed to years | ONS |
| US | Retail Trade, NAICS 44–45, monthly, $m (MRTSSM44000USN) | US Census Bureau via FRED |
| Taiwan | 零售業營業額 (total retail sales), monthly, NT$100m | MOEA |

Totals are converted to pounds at each year's average exchange rate (FRED AEXUSUK, AEXTAUS). Growth is the compound annual growth rate (CAGR) from 2019, the last year before the pandemic, to 2025, the latest full year, in **local currency**, so exchange-rate moves do not distort it. Figures are nominal (not adjusted for inflation), and the three definitions differ: the US total includes car dealers and petrol stations; the UK total includes fuel but not vehicles.

In [10]:
YEARS = (2019, 2025)

# UK: ONS publishes the AVERAGE WEEKLY value (£m) for each month -> weekly value x weeks in the month, summed over the year
ukv = pd.read_csv(f"{RAW}/uk_retail_weekly_value_JE4W.csv", header=None, names=["period", "value"])
ukv = ukv[ukv["period"].astype(str).str.match(r"^\d{4} [A-Z]{3}$", na=False)].copy()
ukv["date"] = pd.to_datetime(ukv["period"], format="%Y %b")
ukv["month_total"] = pd.to_numeric(ukv["value"]) * ukv["date"].dt.days_in_month / 7
uk_year = ukv.groupby(ukv["date"].dt.year)["month_total"].agg(["sum", "count"])
uk_year = uk_year[uk_year["count"] == 12]["sum"]                                      # £m a year, complete years only

# US: Census monthly retail trade ($m) -> sum of 12 months
usv = pd.read_csv(f"{RAW}/us_retail_trade_MRTSSM44000USN.csv")
usv.columns = ["date", "value"]
usv["year"] = pd.to_datetime(usv["date"]).dt.year
us_year = usv.groupby("year")["value"].agg(["sum", "count"])
us_year = us_year[us_year["count"] == 12]["sum"]                                      # $m, complete years only

# Taiwan: MOEA monthly total retail sales (NT$100m), tw_all from section 7
tw_year = tw_all.groupby("year")[2].agg(["sum", "count"])
tw_year = tw_year[tw_year["count"] == 12]["sum"].astype(float) * 100                  # NT$m, complete years only

# Exchange rates: annual averages
def fred_annual(path):
    t = pd.read_csv(path)
    t.columns = ["date", "rate"]
    return t.assign(year=pd.to_datetime(t["date"]).dt.year).set_index("year")["rate"]
usd_per_gbp = fred_annual(f"{RAW}/fx_usd_per_gbp_AEXUSUK.csv")
twd_per_usd = fred_annual(f"{RAW}/fx_twd_per_usd_AEXTAUS.csv")

rows = []
for market, local, unit, to_gbp in [
        ("UK", uk_year, "GBP", lambda y: 1.0),
        ("US", us_year, "USD", lambda y: 1 / usd_per_gbp[y]),
        ("Taiwan", tw_year, "TWD", lambda y: 1 / (twd_per_usd[y] * usd_per_gbp[y]))]:
    start, end = YEARS
    rows.append({"market": market, "currency": unit,
                 f"sales_{start}_local_m": local[start], f"sales_{end}_local_m": local[end],
                 f"sales_{end}_gbp_bn": local[end] * to_gbp(end) / 1000,
                 "cagr_local_pct": ((local[end] / local[start]) ** (1 / (end - start)) - 1) * 100})
market_size = pd.DataFrame(rows)
latest = online.sort_values("date").groupby("market").tail(1).set_index("market")["online_share_pct"]
market_size["online_share_latest_pct"] = market_size["market"].map(latest)
market_size["online_sales_gbp_bn"] = market_size[f"sales_{YEARS[1]}_gbp_bn"] * market_size["online_share_latest_pct"] / 100
market_size.round(2).to_csv(f"{CLEAN}/market_size_by_country.csv", index=False, lineterminator="\n")
market_size.round(1)

,market,currency,sales_2019_local_m,sales_2025_local_m,sales_2025_gbp_bn,cagr_local_pct,online_share_latest_pct,online_sales_gbp_bn
0,UK,GBP,393770.8,483245.2,483.2,3.5,27.9,134.825419
1,US,USD,5329584.0,7318206.0,5547.5,5.4,17.1,948.615241
2,Taiwan,TWD,3926637.8,4844788.0,117.8,3.6,14.214277,16.749571
